In [0]:
%pip install phonenumbers

In [0]:
%restart_python

In [0]:
from pyspark.sql import Column, DataFrame
from pyspark.sql.types import StringType, BooleanType, IntegerType, DecimalType
from pyspark.sql.window import Window
import phonenumbers
import pandas as pd
from pyspark.sql.functions import (
    col, trim, regexp_replace, lower, when, coalesce, lit, 
    to_timestamp, length, locate, initcap, upper, udf, expr, row_number, desc, concat, current_timestamp, regexp_extract
)

In [0]:

def clean_spaces(col_name: str) -> Column:
    return trim(regexp_replace(col(col_name), r'\s+', ' '))

def clean_email(col_name: str) -> Column:
    return lower(trim(col(col_name)))

def is_verified_email(col_name: str) -> Column:
    email_expression = r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"
    return when(col(col_name).isNotNull(),col(col_name).rlike(email_expression)).otherwise(None)

def clean_phone(col_name: str) -> Column:
    return expr(f"regexp_replace(col(col_name), '[^0-9+]', '')")
# phone verify helping function
@udf(returnType=BooleanType())
def is_verified_phone_fn(phone_series: pd.Series) -> pd.Series:
    def check_valid(num):

        if pd.isna(num) or not str(num).strip():
            return False
        
        num_str = str(num).strip()
        if num_str.startswith("00"):
            num_str = "+" + num_str[2:]

        try :
            parsed = phonenumbers.parse(num_str, "US")
            return phonenumbers.is_possible_number(parsed)
        except :
            return False
        
    return pd.Series(phone_series.apply(check_valid))

# phone format helping function
@udf(returnType=StringType())
def format_phone_e164(phone_series: pd.Series) -> pd.Series:
    def format_e164(num):
        if pd.isna(num) or not str(num).strip():
            return None
            
        num_str = str(num).strip()
        if num_str.startswith("00"):
            num_str = "+" + num_str[2:]
            
        try:
            parsed = phonenumbers.parse(num_str, "US")
            if phonenumbers.is_possible_number(parsed):
                return phonenumbers.format_number(parsed, phonenumbers.PhoneNumberFormat.E164)
        except:
            pass

        return None
    return pd.Series(phone_series.apply(format_e164))

# Integer Casting Helping Fuction 
def cast_to_integer(col_name: str, default_val: int =0) -> Column:
    return coalesce(col(col_name).cast(IntegerType()), lit(default_val).cast(IntegerType()))

# Decimal casting Helping Function
def cast_to_decimal(col_name: str , precision: int =10, scale: int =2, default_val: float=0.0) -> Column:
    target_type = DecimalType(precision, scale)
    return coalesce(col(col_name).cast(target_type), lit(default_val).cast(target_type))

# Date standardization Function
def standardize_date(col_name: str) -> Column:
    date_formats = [
        "yyyy-MM-dd'T'HH:mm:ss",
        "yyyy-MM-dd"
    ]
    return coalesce(*[to_timestamp(col(col_name),f) for f in date_formats])

# text_cloumn clean Helping Function
def clean_text_standard(col_name: str, enforce_title_case: bool= True, is_acronym: bool = False) -> Column:
    cleaned_text = trim(regexp_replace(col(col_name), r'\s+', ' '))
    if is_acronym :
        return when((length(cleaned_text) <=5) & (locate(" ", cleaned_text) == 0 ), upper(cleaned_text)).otherwise(initcap(cleaned_text))
    if enforce_title_case:
        return initcap(cleaned_text)
    return cleaned_text

In [0]:
def bronze_sql_customers_silver(df_bronze: DataFrame) -> DataFrame:
    window_spec = Window.partitionBy("customer_id").orderBy(desc("ingesttime"))
    df_dup = (
        df_bronze
        .select("*", row_number().over(window_spec).alias("row_num"))
        .filter(col("row_num") == 1)
        .drop("row_num")
    )
    df_stage_1 =df_dup.withColumns({
        "_ingest_timestamp": standardize_date("ingesttime"),
        "_source_file": col("file_name"),
        "customer_id": upper(trim(col("customer_id"))),
        "first_name": clean_text_standard("first_name"),
        "last_name": clean_text_standard("last_name"),
        "city": clean_text_standard("city"),
        "customer_segment": clean_text_standard("customer_segment"),
        "address": clean_text_standard("address", enforce_title_case=False),
        "region": clean_text_standard("region", is_acronym = True),
        "join_timestamp": standardize_date("signup_date"),
        "email": clean_email("email"),
        "cleaned_phone": format_phone_e164("phone")
        })

    df_finnal = (df_stage_1.withColumns({
        "full_name": concat(col("first_name"), lit(" "), col("last_name")),
        "verified_email": is_verified_email("email"),
        "verified_phone": is_verified_phone_fn("cleaned_phone"),
        "_record_source": lit("SQL_SERVER_CUSTOMERS"),
        "_silver_processed_at": current_timestamp()
        })
    )

    df_silver = (df_finnal.drop("phone", "signup_date", "file_name", "ingesttime", "_rescued_data"))

    return df_silver

In [0]:
def bronze_crm_customers_silver(df_bronze: DataFrame) -> DataFrame:
    window_spec = Window.partitionBy("customer_id").orderBy(desc("ingesttime"))
    df_dedup = (
        df_bronze.select("*", row_number().over(window_spec).alias("row_num"))
        .filter(col("row_num") == 1)
        .drop("row_num")
    )

    df_stage_1 = (df_dedup
                  .withColumns({
                      "crm_last_updated": standardize_date("crm_last_updated"),
                      "customer_id": upper(trim(col("customer_id"))),
                      "last_compaign_engaged": clean_text_standard("last_compaign_engaged"),
                      "lifetime_value_estimate": cast_to_decimal("life_time_value_estimate", precision=10, scale=2, default_val=0.0),
                      "churn_risk_score": cast_to_decimal("churn_risk_score", 5, 3, 0.0),
                      "loyalty_tier": clean_text_standard("loyalty_tier", is_acronym = False),
                      "marketing_opt_in": coalesce(col("marketing_opt_in").cast(BooleanType()), lit(False)),
                      "preferred_channel" : clean_text_standard("preferred_channel", is_acronym= False),
                      "_ingestion_timestamp": standardize_date("ingesttime"),
                      "_source_file": col("file_name")
                      })
                )
    df_silver = (df_stage_1.withColumns({
                      "_record_source": lit("CRM_SYSTEM_CUSTOMERS"),
                      "_silver_processed_at": current_timestamp()
                      })
                )
    df_finnal = (df_silver.drop("ingesttime", "file_name", "crm_last_updated", "_rescued_data"))
    return df_finnal

In [0]:
def bronze_sql_products_silver(df_bronze: DataFrame) -> DataFrame:
    window_spec = Window.partitionBy("product_id").orderBy(desc("ingesttime"))
    df_dedup = (
        df_bronze.select("*", row_number().over(window_spec).alias("row_num"))
                    .filter(col("row_num") == 1)
                    .drop("row_num")
    )

    df_stage_1 = (df_dedup
                  .withColumns({
                      "_ingestion_timestamp": col("ingesttime"),
                      "_source_file": col("file_name"),
                      "product_id": upper(trim(col("product_id"))),
                      "product_name": clean_text_standard("product_name", enforce_title_case=True),
                      "category": clean_text_standard("category", enforce_title_case=True),
                      "subcategory": clean_text_standard("subcategory", enforce_title_case= True),
                      "brand" : clean_text_standard("brand", enforce_title_case=False),
                      "unit_price": cast_to_decimal("unit_price", precision=10, scale=2, default_val=0.0),
                      "cost_price": cast_to_decimal("cost_price", precision=10, scale=2, default_val=0.0),
                      "stock_quantity": cast_to_integer("stock_quantity", default_val=0),
                      "reorder_threshold": cast_to_integer("reorder_theshold", default_val=0),
                      "supplier_id": upper(trim(col("supplier_id"))),
                      "created_date": standardize_date("created_date")
                  }))
    df_silver = (df_stage_1.withColumns({
                    "is_margin_positive": col("unit_price") >= col("cost_price"),
                    "is_reorder_needed": col("stock_quantity") <= col("reorder_threshold"),
                    "_record_source": lit("SQL_SERVER_PRODUCTS"),
                    "_silver_processed_at": current_timestamp()
                    })
    )

    df_finnal = df_silver.drop("ingesttime", "file_name", "_rescued_data")
    return df_finnal

In [0]:
def bronze_sql_sale_transactions(df_bronze: DataFrame) -> DataFrame:
    window_spec = Window.partitionBy("transaction_id").orderBy(desc("last_modified_ts"),desc("ingesttime"))
    df_dedup = (
        df_bronze.select("*", row_number().over(window_spec).alias("row_num"))
                    .filter(col("row_num") == 1)
                    .drop("row_num")
    )

    df_stage_1 = (df_dedup.withColumns({
        "_ingestion_timestamp": col("ingesttime"),
        "_source_file": col("file_name"),
        "transaction_id": upper(trim(col("transaction_id"))),
        "customer_id": upper(trim(col("customer_id"))),
        "product_id": upper(trim(col("product_id"))),
        "store_id": upper(trim(col("store_id"))),
        "payment_method": upper(trim(col("payment_method"))) ,
        "quantity": cast_to_integer("quantity"),
        "unit_price": cast_to_decimal("unit_price", precision = 10, scale = 2, default_val = 0.0),
        "discount_pct": cast_to_decimal("discount_pct", precision = 10, scale = 2, default_val = 0.0),
        "total_amount": cast_to_decimal("total_amount", precision = 10, scale = 2, default_val = 0.0),
        "region": clean_text_standard("region", is_acronym = True),
        "transaction_timestamp": standardize_date("transaction_ts"),
        "last_modified_timestamp": standardize_date("last_modified_ts"),
    }))

    calculated_amount = (col("quantity") * col("unit_price") * (lit(1.0)- col("discount_pct")))

    df_stage_2 = (df_stage_1.withColumns({
        "is_pricing_correct": col("total_amount") == calculated_amount.cast("decimal(10,2)"),
        "is_time_valid": col("last_modified_timestamp") >= col("transaction_timestamp"),
        "_record_source": lit("SQL_SERVER_SALES"),
        "_silver_processed_at": current_timestamp()
    }))

    df_finnal = df_stage_2.drop("ingesttime", "file_name", "_rescued_data", "transaction_ts", "last_modified_ts")
    return df_finnal

In [0]:
def bronze_clickstream(df_bronze: DataFrame) -> DataFrame:
    window_spec = Window.partitionBy("event_id").orderBy(desc("ingesttime"))
    df_dedup = (
        df_bronze.select("*", row_number().over(window_spec).alias("row_num"))
                    .filter(col("row_num") == 1)
                    .drop("row_num")
    )

    allowed_events = ["PAGE_VIEW", "SEARCH", "ADD_TO_CART", "REMOVE_FROM_CART", "PURCHASE"]
    allowed_devices = ["DESKTOP", "MOBILE", "TABLET"]

    raw_customer_id = trim(col("customer_id"))

    df_stage_1 = (df_dedup.withColumns({
        "_ingestion_timestamp": col("ingesttime"),
        "_source_file": col("file_name"),
        "customer_id": when(lower(raw_customer_id) == "null",lit(None).cast("string")).otherwise(upper(raw_customer_id)),
        "event_id": upper(trim(col("event_id"))),
        "session_id": lower(trim(col("session_id"))),
        "product_id": upper(trim(col("product_id"))),

        "event_type": upper(trim(col("event_type"))),
        "device_type": upper(trim(col("device_type"))),
        "page_url": trim(col("page_url")),
        "event_timestamp": standardize_date("timestamp")
        })
    )

    url_query_extractor = r"[?&]q=([^&]+)"
    extracted_query = regexp_extract(col("page_url"), url_query_extractor, 1)

    df_silver = (df_stage_1.withColumns({
        "is_event_valid": col("event_type").isin(allowed_events),
        "is_device_valid": col("device_type").isin(allowed_devices),
        "search_query": when(col("event_type") == "SEARCH",
                             when(extracted_query != "", extracted_query).otherwise(None)
                             ).otherwise(None),
        "_record_source": lit("WEB_CLICKSTREAM"),
        "_silver_processed_at": current_timestamp()
    }))
    df_finnal = df_silver.drop("ingesttime", "file_name", "_rescued_data", "timestamp")
    return df_finnal